## Assignment 1:

- Using Numpy to implement the soft-margin SVM model. 
- Train this model using SGD method on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.

In [7]:
from pathlib import Path
import numpy as np
from PIL import Image


class SoftMarginSVM:
    def __init__(self, lr=1e-7, lambda_reg=1e-4, epochs=8, batch_size=32, random_state=42):
        self.lr = lr
        self.lambda_reg = lambda_reg
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.w = None
        self.b = 0.0

    def fit(self, X, y, verbose=True):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=np.float32)
        self.b = 0.0

        rng = np.random.default_rng(self.random_state)

        for epoch in range(1, self.epochs + 1):
            indices = rng.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for start in range(0, n_samples, self.batch_size):
                end = min(start + self.batch_size, n_samples)
                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]

                margins = y_batch * (X_batch @ self.w + self.b)
                active = margins < 1

                if np.any(active):
                    grad_w = self.lambda_reg * self.w - np.mean(
                        y_batch[active, None] * X_batch[active], axis=0
                    )
                    grad_b = -np.mean(y_batch[active])
                else:
                    grad_w = self.lambda_reg * self.w
                    grad_b = 0.0

                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b

            if verbose:
                hinge = np.maximum(0.0, 1.0 - y * (X @ self.w + self.b))
                loss = 0.5 * self.lambda_reg * np.dot(self.w, self.w) + np.mean(hinge)
                print(f"Epoch {epoch:02d}/{self.epochs} - loss: {loss:.6f}")

    def decision_function(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, -1)


dataset_root = Path(r"D:\UIT\ds102\lap3\dataset\chest_xray")
classes = {"NORMAL": -1, "PNEUMONIA": 1}
valid_ext = {".jpeg"}


def load_split(split_name, image_size=(128, 128)):
    X_list = []
    y_list = []

    for class_name, label in classes.items():
        class_dir = dataset_root / split_name / class_name
        image_paths = [p for p in class_dir.iterdir() if p.suffix.lower() in valid_ext]

        for image_path in image_paths:
            img = Image.open(image_path).convert("L")
            img = img.resize(image_size)
            x = np.asarray(img, dtype=np.float32).reshape(-1)

            X_list.append(x)
            y_list.append(label)

    X = np.stack(X_list, axis=0)
    y = np.asarray(y_list, dtype=np.int32)

    return X, y


def zscore_fit_transform(X_train):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    X_train_z = (X_train - mean) / std
    return X_train_z.astype(np.float32), mean.astype(np.float32), std.astype(np.float32)


def zscore_transform(X, mean, std):
    return ((X - mean) / std).astype(np.float32)


X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

X_train, train_mean, train_std = zscore_fit_transform(X_train)
X_val = zscore_transform(X_val, train_mean, train_std)
X_test = zscore_transform(X_test, train_mean, train_std)

svm = SoftMarginSVM(lr=1e-7, lambda_reg=1e-4, epochs=8, batch_size=32, random_state=42)
svm.fit(X_train, y_train, verbose=True)

y_pred_test = svm.predict(X_test)


def precision_recall_f1(y_true, y_pred, positive_label=1):
    tp = np.sum((y_true == positive_label) & (y_pred == positive_label))
    fp = np.sum((y_true != positive_label) & (y_pred == positive_label))
    fn = np.sum((y_true == positive_label) & (y_pred != positive_label))

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    return precision, recall, f1


precision, recall, f1 = precision_recall_f1(y_test, y_pred_test, positive_label=1)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

Epoch 01/8 - loss: 0.992698
Epoch 02/8 - loss: 0.985396
Epoch 03/8 - loss: 0.978095
Epoch 04/8 - loss: 0.970793
Epoch 05/8 - loss: 0.963491
Epoch 06/8 - loss: 0.956189
Epoch 07/8 - loss: 0.948887
Epoch 08/8 - loss: 0.941586
Precision: 0.9399
Recall   : 0.6821
F1-score : 0.7905


## Assigment 2:
- Implement SVM method using a machine learning library (such as sklearn or sktorch).
- Train this model on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.
- Compare the results of SVM using library to those of implemented SVM.

In [9]:
from time import perf_counter
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

def l2_normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms < 1e-8, 1.0, norms)
    return (X / norms).astype(np.float32)


X_train_lib, y_train_lib = load_split("train")
X_val_lib, y_val_lib = load_split("val")
X_test_lib, y_test_lib = load_split("test")

X_train_lib, mean_lib, std_lib = zscore_fit_transform(X_train_lib)
X_val_lib = zscore_transform(X_val_lib, mean_lib, std_lib)
X_test_lib = zscore_transform(X_test_lib, mean_lib, std_lib)

X_train_lib = l2_normalize_rows(X_train_lib)
X_val_lib = l2_normalize_rows(X_val_lib)
X_test_lib = l2_normalize_rows(X_test_lib)

epochs = 8
batch_size = 64
alpha = 1e-3
eta0 = 1e-2

svm_lib = SGDClassifier(
    loss="hinge",
    penalty="l2",
    alpha=alpha,
    learning_rate="constant",
    eta0=eta0,
    average=True,
    random_state=42,
)

class_values = np.array(sorted(set(classes.values())), dtype=np.int32)
rng = np.random.default_rng(42)
n_samples = X_train_lib.shape[0]
first_update = True

start_time = perf_counter()
for epoch in range(1, epochs + 1):
    indices = rng.permutation(n_samples)
    X_shuffled = X_train_lib[indices]
    y_shuffled = y_train_lib[indices]

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        X_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]

        if first_update:
            svm_lib.partial_fit(X_batch, y_batch, classes=class_values)
            first_update = False
        else:
            svm_lib.partial_fit(X_batch, y_batch)

    scores_train = svm_lib.decision_function(X_train_lib)
    hinge = np.maximum(0.0, 1.0 - y_train_lib * scores_train)
    loss = 0.5 * alpha * float(np.dot(svm_lib.coef_.ravel(), svm_lib.coef_.ravel())) + float(np.mean(hinge))
    print(f"Epoch {epoch:02d}/{epochs} - loss: {loss:.6f}")

train_time_lib = perf_counter() - start_time

y_pred_lib = svm_lib.predict(X_test_lib)
precision_lib = precision_score(y_test_lib, y_pred_lib, pos_label=1, zero_division=0)
recall_lib = recall_score(y_test_lib, y_pred_lib, pos_label=1, zero_division=0)
f1_lib = f1_score(y_test_lib, y_pred_lib, pos_label=1, zero_division=0)


if all(name in globals() for name in ["precision", "recall", "f1"]):
    print(f"A1 Precision: {precision:.4f} | A2 Precision: {precision_lib:.4f}")
    print(f"A1 Recall   : {recall:.4f} | A2 Recall   : {recall_lib:.4f}")
    print(f"A1 F1-score : {f1:.4f} | A2 F1-score : {f1_lib:.4f}")
else:
    print("\nRun Cell 2 first if you want automatic comparison with Assignment 1 metrics.")

Epoch 01/8 - loss: 0.478784
Epoch 02/8 - loss: 0.420362
Epoch 03/8 - loss: 0.396125
Epoch 04/8 - loss: 0.381910
Epoch 05/8 - loss: 0.372414
Epoch 06/8 - loss: 0.365911
Epoch 07/8 - loss: 0.361091
Epoch 08/8 - loss: 0.357285
A1 Precision: 0.9399 | A2 Precision: 0.8641
A1 Recall   : 0.6821 | A2 Recall   : 0.8641
A1 F1-score : 0.7905 | A2 F1-score : 0.8641
